In [ ]:
import os
import numpy as np
from numba import njit
# Set JAX platform before import
# _JAX_DEVICE = os.environ.get("JAX_PLATFORMS", "cpu")
# if "JAX_PLATFORMS" not in os.environ:
#     os.environ["JAX_PLATFORMS"] = _JAX_DEVICE

import jax
import jax.numpy as jnp
import jax.random as jr
from jax import vmap
from dynamax.hidden_markov_model import CategoricalHMM

In [ ]:
@njit
def sample_numba(
    n_chains: int,
    n_rounds: int,
    initial_probs: np.ndarray,      # shape (2,)
    transition_matrix: np.ndarray,  # shape (2, 2)
    emission_probs: np.ndarray,     # shape (2, 4)
) -> np.ndarray:
    samples = np.empty((n_chains, n_rounds), dtype=np.int8)

    for i in range(n_chains):
        u = np.random.rand()
        if u < initial_probs[0]:
            state = 0
        else:
            state = 1
        
        for j in range(n_rounds):
            v = np.random.rand()
            cumulative = 0.0
            for k in range(4):
                cumulative += emission_probs[state, k]
                if v < cumulative:
                    break
            samples[i, j] = k

            w = np.random.rand()
            if w < transition_matrix[state, 0]:
                state = 0
            else:
                state = 1

    return samples

def sample_dynamax(
        n_chains: int,
        n_rounds: int,
        hmm: CategoricalHMM,
        params: dict,
    ):
    key = jr.PRNGKey(np.random.randint(0, 2**32))
    keys = jr.split(key, n_chains)
    _, samples = vmap(lambda k: hmm.sample(params, k, n_rounds))(keys)
    
    return np.array(samples)


In [ ]:
n_chains = 1000
n_rounds = 50

a = 0.01
b = 0.1

emissions = [
    [1.0, 0.0, 0.0, 0.0],  # stormy state
    [0.25, 0.25, 0.25, 0.25],  # calm state
]

pi_a = a / (a + b)  # stationary prob. of being in stormy state
pi_b = b / (a + b)  # stationary prob. of being in calm state
initial_probs = np.array([pi_b, pi_a])
transition_matrix=np.array([[1.0 - a, a], [b, 1.0 - b]])
emission_probs=np.array(emissions)

hmm = CategoricalHMM(num_states=2, emission_dim=1, num_classes=4)
params, _ = hmm.initialize(
    initial_probs=jnp.array(initial_probs),
    transition_matrix=jnp.array(transition_matrix),
    emission_probs=jnp.array(emission_probs.reshape(2, 1, 4)),
    )

In [ ]:
sample_numba(n_chains, n_rounds, initial_probs, transition_matrix, emission_probs)
%timeit sample_numba(n_chains, n_rounds, initial_probs, transition_matrix, emission_probs)

In [ ]:
# sample_dynamax(n_chains, n_rounds, hmm, params)
# %timeit sample_dynamax(n_chains, n_rounds, hmm, params)